In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/My Drive/ussf_ssac_23_soccer_gnn')

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'  # force Keras 2 API — Spektral requires it
!pip install tf-keras spektral ipywidgets progressbar2 -q

In [ ]:
import copy, logging, pickle, random, sys, time
from collections import defaultdict
from os.path import isfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import progressbar
import tensorflow as tf

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    log_loss as sk_log_loss, roc_curve, precision_recall_curve
)
from spektral.data import Dataset, Graph, DisjointLoader
from spektral.layers import CrystalConv, GATConv, GlobalAvgPool
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

logger = logging.getLogger('gat_exp')
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler(sys.stdout))

print('TF version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## GAT vs CrystalConv — 2×2 Experiment

| | **Balanced (BCE)** | **Imbalanced (Focal Loss)** |
|---|---|---|
| **CrystalConv** | Baseline — replicate paper | Architecture fixed, data changed |
| **GAT** | Architecture changed, data fixed | **Headline result** |

**Key differences from paper:**
- GAT learns *per-edge attention weights* from node features instead of aggregating uniformly
- Focal Loss (γ=2, α=0.25) trains on the full imbalanced dataset (~95% negative) instead of a downsampled 50/50 set
- PR-AUC added as primary metric for imbalanced evaluation alongside ROC-AUC and Log-Loss

**Note on success definition:** The balanced dataset labels a sequence successful if it *reaches the penalty area*; the imbalanced dataset labels it successful if it *leads to a goal*. Results across columns are therefore not directly comparable and should be interpreted separately.

### Configuration

In [ ]:
# ── Experiment config ─────────────────────────────────────────────────
GENDER        = 'men'         # 'men' or 'women'
DATASET_TYPE  = 'both'        # 'balanced', 'imbalanced', or 'both'
MATRIX_TYPE   = 'normal'
LEARNING_RATE = 1e-3
EPOCHS        = 150
BATCH_SIZE    = 16
SEED          = 15

# CrystalConv architecture (matches original paper)
CC_CHANNELS = 128
CC_LAYERS   = 3

# GAT architecture
GAT_CHANNELS = 128   # output channels (after head averaging)
GAT_HEADS    = 4
GAT_LAYERS   = 3

# Focal Loss
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

# ── Feature selection ─────────────────────────────────────────────────
# Node feature order (12 total):
#   0:x  1:y  2:vx  3:vy  4:v  5:v_angle  6:dist_goal  7:angle_goal
#   8:dist_ball  9:angle_ball  10:att_flag  11:pot_receiver
NODE_FEATURE_IDXS = [0, 1, 4, 5, 6, 7, 8, 9, 10]  # drops vx, vy, pot_receiver

# Edge feature order (6 total):
#   0:distance  1:speed_diff  2:pos_sin  3:pos_cos  4:vel_sin  5:vel_cos
EDGE_FEATURE_IDXS = [0, 2, 3, 4, 5]  # drops speed_diff

### Data Loading

In [ ]:
S3 = 'https://ussf-ssac-23-soccer-gnn.s3.us-east-2.amazonaws.com/public/counterattack/'

def get_data(file_name):
    if not isfile(file_name):
        url = S3 + file_name
        logger.info(f'Downloading {url} ...')
        r = requests.get(url, stream=True)
        file_size = int(r.headers.get('Content-Length', 0))
        chunk = 1024 * 1000
        n_bars = max(1, int(np.ceil(file_size / chunk)))
        bar = progressbar.ProgressBar(maxval=n_bars).start()
        with open(file_name, 'wb') as f:
            for i, block in enumerate(r.iter_content(chunk_size=chunk)):
                f.write(block)
                bar.update(min(i + 1, n_bars))
                time.sleep(0.01)
        bar.finish()
    with open(file_name, 'rb') as f:
        return pickle.load(f)

balanced_data, imbalanced_data = None, None

if DATASET_TYPE in ('balanced', 'both'):
    print(f'Downloading balanced ({GENDER}) ...')
    balanced_data = get_data(f'{GENDER}.pkl')
    print('Balanced samples:', len(balanced_data['binary']))

if DATASET_TYPE in ('imbalanced', 'both'):
    print(f'\nDownloading imbalanced ({GENDER}) ...')
    imbalanced_data = get_data(f'{GENDER}_imbalanced.pkl')
    n_imb = len(imbalanced_data['label'])
    pct_pos = np.mean([y[0] for y in imbalanced_data['label']])
    print(f'Imbalanced samples: {n_imb:,}  |  positive rate: {pct_pos:.1%}')

### Spektral Dataset Classes

In [ ]:
def _slice_features(x_list, e_list, node_idxs, edge_idxs):
    if node_idxs is not None:
        x_list = [x[:, node_idxs] for x in x_list]
    if edge_idxs is not None:
        e_list = [e[:, edge_idxs] for e in e_list]
    return x_list, e_list


class BalancedCounterDataset(Dataset):
    def __init__(self, data, matrix_type, node_idxs=None, edge_idxs=None, **kwargs):
        self._raw   = data
        self._mtype = matrix_type
        self._nidx  = node_idxs
        self._eidx  = edge_idxs
        super().__init__(**kwargs)

    def read(self):
        mat = self._raw[self._mtype]
        xs, es = _slice_features(mat['x'], mat['e'], self._nidx, self._eidx)
        return [
            Graph(x=x, a=a, e=e, y=y)
            for x, a, e, y in zip(xs, mat['a'], es, self._raw['binary'])
        ]


class ImbalancedCounterDataset(Dataset):
    """Wraps a pre-split dict {a, x, e, label} produced by split_by_sequences."""
    def __init__(self, split_data, node_idxs=None, edge_idxs=None, **kwargs):
        self._raw  = split_data
        self._nidx = node_idxs
        self._eidx = edge_idxs
        super().__init__(**kwargs)

    def read(self):
        xs, es = _slice_features(
            list(self._raw['x']), list(self._raw['e']), self._nidx, self._eidx
        )
        return [
            Graph(x=x, a=a, e=e, y=y)
            for x, a, e, y in zip(xs, self._raw['a'], es, self._raw['label'])
        ]

### Sequence-Aware Train/Test Split (Imbalanced Data)

Each counterattacking *sequence* contains multiple frames. A naive random split risks putting frames from the same sequence in both train and test, leaking future-frame information. We split at the sequence level, stratified by outcome.

In [ ]:
def split_by_sequences(data, test_pct=0.20, seed=42):
    """
    Stratified sequence-level train/test split for the imbalanced dataset.
    All frames from the same sequence land in the same split.
    """
    keys = ['a', 'x', 'e', 'id', 'label']
    train_set, test_set = defaultdict(list), defaultdict(list)
    labels_flat = np.array([y[0] for y in data['label']])

    for outcome in np.unique(labels_flat):
        outcome_idxs = np.where(labels_flat == outcome)[0]
        target_test_n = int(test_pct * len(outcome_idxs))
        sub = {k: data[k][outcome_idxs] for k in keys}

        seq_ids = sorted({int(s) for s in sub['id']})
        rng = random.Random(seed)
        rng.shuffle(seq_ids)

        test_seq_ids, test_so_far = set(), 0
        for sid in seq_ids:
            if test_so_far >= target_test_n:
                break
            test_seq_ids.add(sid)
            test_so_far += int(np.sum(sub['id'] == sid))

        for idx in range(len(sub['label'])):
            dst = test_set if int(sub['id'][idx]) in test_seq_ids else train_set
            for k in keys:
                dst[k].append(sub[k][idx])

    tr_pos = np.mean([y[0] for y in train_set['label']])
    te_pos = np.mean([y[0] for y in test_set['label']])
    print(f'Train: {len(train_set["label"]):,} frames | {tr_pos:.1%} positive')
    print(f'Test:  {len(test_set["label"]):,} frames | {te_pos:.1%} positive')
    return train_set, test_set


imb_train, imb_test = None, None
if DATASET_TYPE in ('imbalanced', 'both'):
    print('Splitting imbalanced dataset by sequence ...')
    imb_train, imb_test = split_by_sequences(imbalanced_data, test_pct=0.20, seed=SEED)

### Model Definitions

In [ ]:
class CrystalGNN(Model):
    """Replication of the original paper's architecture."""
    def __init__(self, n_layers=3, channels=128):
        super().__init__()
        self.convs  = [CrystalConv() for _ in range(n_layers)]
        self.pool   = GlobalAvgPool()
        self.dense1 = Dense(channels, activation='relu')
        self.drop1  = Dropout(0.5)
        self.dense2 = Dense(channels, activation='relu')
        self.drop2  = Dropout(0.5)
        self.out    = Dense(1, activation='sigmoid')

    def call(self, inputs, training=False):
        x, a, e, i = inputs
        for conv in self.convs:
            x = conv([x, a, e])
        x = self.pool([x, i])
        x = self.dense1(x)
        x = self.drop1(x, training=training)
        x = self.dense2(x)
        x = self.drop2(x, training=training)
        return self.out(x)


class GATGNN(Model):
    """
    GAT-based GNN. Replaces CrystalConv's uniform aggregation with learned
    per-edge attention weights computed from connected node feature pairs.

    Standard GAT attention is a function of the source and target node features
    (position, velocity, distance-to-goal, etc.) — edge features are not used
    in the attention computation. The model dynamically weights each player
    relationship based purely on the numerical state of those two players.
    """
    def __init__(self, n_layers=3, channels=128, attn_heads=4):
        super().__init__()
        # Each layer gets a unique seed so weights are not identical at init
        self.convs = [
            GATConv(
                channels,
                attn_heads=attn_heads,
                concat_heads=False,
                dropout_rate=0.5,
                return_attn_coef=False,
                kernel_initializer=tf.keras.initializers.GlorotUniform(seed=i),
                attn_kernel_initializer=tf.keras.initializers.GlorotUniform(seed=i + 100),
            )
            for i in range(n_layers)
        ]
        self.pool   = GlobalAvgPool()
        self.dense1 = Dense(channels * 2, activation='relu')
        self.drop1  = Dropout(0.5)
        self.dense2 = Dense(channels, activation='relu')
        self.drop2  = Dropout(0.5)
        self.out    = Dense(1, activation='sigmoid')

    def call(self, inputs, training=False):
        x, a, e, i = inputs   # e is unused: standard GAT operates on node features only
        for conv in self.convs:
            x = conv([x, a])
        x = self.pool([x, i])
        x = self.dense1(x)
        x = self.drop1(x, training=training)
        x = self.dense2(x)
        x = self.drop2(x, training=training)
        return self.out(x)

    def get_attention_weights(self, inputs):
        """
        Return attention coefficients from every GAT layer for a single graph.
        Must be called in eager mode (not inside @tf.function).
        """
        x, a, e, i = inputs
        attn_per_layer = []
        for conv in self.convs:
            conv.return_attn_coef = True
            x, attn = conv([x, a])
            attn_per_layer.append(attn.numpy())
            conv.return_attn_coef = False
        return attn_per_layer

### Focal Loss

Down-weights easy negatives so the model focuses on the harder rare positives. Standard parameters from the RetinaNet paper (Lin et al., 2017): γ=2, α=0.25.

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce     = -(y_true * tf.math.log(y_pred)
                    + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        p_t     = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        weight  = alpha * y_true + (1.0 - alpha) * (1.0 - y_true)
        return tf.reduce_mean(weight * tf.pow(1.0 - p_t, gamma) * bce)
    return loss

### Training & Evaluation Infrastructure

In [ ]:
def evaluate(model, loader_te):
    y_true, y_pred = [], []
    for batch in loader_te:
        inputs, target = batch
        p = model(inputs, training=False)
        y_true.append(np.array(target))
        y_pred.append(np.array(p))
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)
    return {
        'roc_auc':  round(roc_auc_score(y_true, y_pred), 4),
        'pr_auc':   round(average_precision_score(y_true, y_pred), 4),
        'log_loss': round(sk_log_loss(y_true, y_pred), 4),
    }, y_true, y_pred


def train_model(model, loader_tr, loader_te, optimizer, loss_fn, epochs, label=''):
    @tf.function(input_signature=loader_tr.tf_signature(), experimental_relax_shapes=True)
    def train_step(inputs, target):
        with tf.GradientTape() as tape:
            predictions = model(inputs, training=True)
            loss = loss_fn(target, predictions) + sum(model.losses)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    sep = '=' * 55
    print('\n' + sep)
    print('  Training: ' + label)
    print(sep)
    for epoch in range(1, epochs + 1):
        step = total_loss = 0
        for batch in loader_tr:
            step += 1
            total_loss += train_step(*batch)
            if step == loader_tr.steps_per_epoch:
                break
        if epoch % 10 == 0 or epoch == 1:
            avg = total_loss / loader_tr.steps_per_epoch
            print(f'  Epoch {epoch:>3}/{epochs}  loss={avg:.4f}')

    metrics, y_true, y_pred = evaluate(model, loader_te)
    print('\n  ROC-AUC=' + str(metrics['roc_auc'])
          + '  PR-AUC=' + str(metrics['pr_auc'])
          + '  Log-Loss=' + str(metrics['log_loss']))
    return metrics, y_true, y_pred

### Run the 2×2 Experiment Grid

Cells are run sequentially. Each creates a fresh model and fresh loaders. Training ~150 epochs per cell — expect ~5–15 min per run on GPU.

In [ ]:
results = {}
trained_models = {}


def run_cell(arch, data_type):
    key = f'{arch}_{data_type}'

    if data_type == 'balanced':
        dataset = BalancedCounterDataset(
            balanced_data, MATRIX_TYPE,
            node_idxs=NODE_FEATURE_IDXS, edge_idxs=EDGE_FEATURE_IDXS
        )
        idxs    = np.random.RandomState(SEED).permutation(len(dataset))
        split   = int(0.7 * len(dataset))
        ds_tr   = dataset[idxs[:split]]
        ds_te   = dataset[idxs[split:]]
        loss_fn = BinaryCrossentropy()
    else:
        ds_tr   = ImbalancedCounterDataset(
            imb_train, node_idxs=NODE_FEATURE_IDXS, edge_idxs=EDGE_FEATURE_IDXS
        )
        ds_te   = ImbalancedCounterDataset(
            imb_test, node_idxs=NODE_FEATURE_IDXS, edge_idxs=EDGE_FEATURE_IDXS
        )
        loss_fn = focal_loss(FOCAL_GAMMA, FOCAL_ALPHA)

    loader_tr = DisjointLoader(ds_tr, batch_size=BATCH_SIZE, epochs=EPOCHS)
    loader_te = DisjointLoader(ds_te, batch_size=BATCH_SIZE, epochs=1, shuffle=False)

    if arch == 'crystal':
        model = CrystalGNN(n_layers=CC_LAYERS, channels=CC_CHANNELS)
    else:
        model = GATGNN(n_layers=GAT_LAYERS, channels=GAT_CHANNELS, attn_heads=GAT_HEADS)

    optimizer = Adam(LEARNING_RATE)
    loss_label = 'BCE' if data_type == 'balanced' else 'Focal'
    label = f'{arch.upper()} + {loss_label} on {GENDER} {data_type}'

    metrics, y_true, y_pred = train_model(
        model, loader_tr, loader_te, optimizer, loss_fn, EPOCHS, label
    )
    results[key] = {**metrics, 'y_true': y_true, 'y_pred': y_pred}
    trained_models[key] = model


# Experiments to run:
#   balanced   -> GAT + BCE only
#   imbalanced -> CrystalConv + Focal, GAT + Focal
#   both       -> all three
if DATASET_TYPE in ('balanced', 'both'):
    run_cell('gat', 'balanced')

if DATASET_TYPE in ('imbalanced', 'both'):
    run_cell('crystal', 'imbalanced')
    run_cell('gat', 'imbalanced')

### Results

In [ ]:
rows = []
for arch in ['crystal', 'gat']:
    for dtype in ['balanced', 'imbalanced']:
        key = f'{arch}_{dtype}'
        m   = results[key]
        rows.append({
            'Gender':         GENDER.capitalize(),
            'Architecture':   'CrystalConv' if arch == 'crystal' else 'GAT',
            'Dataset':         dtype.capitalize(),
            'Loss':            'BCE' if dtype == 'balanced' else 'Focal',
            'ROC-AUC':         m['roc_auc'],
            'PR-AUC':          m['pr_auc'],
            'Log-Loss':        m['log_loss'],
        })

df = pd.DataFrame(rows).set_index(['Gender', 'Architecture', 'Dataset', 'Loss'])
print(df.to_string())

### Curves

ROC and Precision-Recall curves for all four configurations. PR-AUC is the primary metric for the imbalanced experiments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors   = {'crystal_balanced': 'tab:blue',   'gat_balanced': 'tab:orange',
            'crystal_imbalanced': 'tab:green', 'gat_imbalanced': 'tab:red'}
labels_m = {'crystal_balanced': 'CrystalConv + BCE (balanced)',
            'gat_balanced':     'GAT + BCE (balanced)',
            'crystal_imbalanced': 'CrystalConv + Focal (imbalanced)',
            'gat_imbalanced':   'GAT + Focal (imbalanced)'}

for key, m in results.items():
    yt, yp = m['y_true'], m['y_pred']
    fpr, tpr, _ = roc_curve(yt, yp)
    pre, rec, _ = precision_recall_curve(yt, yp)
    c = colors[key]
    axes[0].plot(fpr, tpr, color=c, label=f'{labels_m[key]} (AUC={m["roc_auc"]})')
    axes[1].plot(rec, pre, color=c, label=f'{labels_m[key]} (AP={m["pr_auc"]})')

axes[0].plot([0,1],[0,1],'k--', linewidth=0.8)
axes[0].set(title='ROC Curve', xlabel='FPR', ylabel='TPR')
axes[0].legend(fontsize=7)

axes[1].set(title='Precision-Recall Curve', xlabel='Recall', ylabel='Precision')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig('curves_2x2.png', dpi=150)
plt.show()

### GAT Attention Analysis

For each test sample we extract per-layer attention weights from the trained GAT (imbalanced). We then compare the distribution of attention on attacking–attacking edges vs attacking–defending edges for successful vs unsuccessful counterattacks.

**Requires** `Attacking Team Flag` to be present in node features (index 10, included by default).

In [ ]:
# att_flag is at original index 10; after slicing NODE_FEATURE_IDXS=[0,1,4,5,6,7,8,9,10]
# it lands at position 8 in the sliced matrix
ATT_FLAG_IDX = NODE_FEATURE_IDXS.index(10) if NODE_FEATURE_IDXS is not None else 10
N_SAMPLES    = 200  # number of test graphs to analyse (keep small — runs in eager mode)

gat_model = trained_models['gat_imbalanced']

ds_te_attn = ImbalancedCounterDataset(
    imb_test, node_idxs=NODE_FEATURE_IDXS, edge_idxs=EDGE_FEATURE_IDXS
)
loader_attn = DisjointLoader(ds_te_attn, batch_size=1, epochs=1, shuffle=False)

att_success   = []   # mean last-layer attention on att-att edges, successful sequences
att_fail      = []   # same for unsuccessful
cross_success = []   # mean attention on att-def / def-att edges, successful
cross_fail    = []

for n_done, batch in enumerate(loader_attn):
    if n_done >= N_SAMPLES:
        break
    inputs, target = batch
    x, a, e, i = inputs
    label = int(target.numpy()[0][0])

    # get attention weights from all layers (eager call — no @tf.function)
    attn_layers = gat_model.get_attention_weights(inputs)
    attn_last   = attn_layers[-1]   # shape (n_edges, n_heads) or (n_edges,)
    if attn_last.ndim > 1:
        attn_last = attn_last.mean(axis=-1)  # average over heads

    # rebuild COO edge list from sparse adjacency
    a_np   = tf.sparse.to_dense(a).numpy() if isinstance(a, tf.SparseTensor) else a.numpy()
    src, dst = np.where(a_np > 0)
    if len(src) == 0 or len(attn_last) == 0:
        continue

    n_edges = min(len(src), len(attn_last))
    src, dst = src[:n_edges], dst[:n_edges]

    x_np      = x.numpy()
    src_att   = x_np[src, ATT_FLAG_IDX]
    dst_att   = x_np[dst, ATT_FLAG_IDX]
    aa_mask   = (src_att == 1) & (dst_att == 1)
    cross_mask = (src_att != dst_att)

    if aa_mask.sum() > 0:
        mean_aa = attn_last[aa_mask].mean()
        if label == 1: att_success.append(mean_aa)
        else:           att_fail.append(mean_aa)

    if cross_mask.sum() > 0:
        mean_cross = attn_last[cross_mask].mean()
        if label == 1: cross_success.append(mean_cross)
        else:           cross_fail.append(mean_cross)

print(f'Analysed {n_done} test graphs')
print(f'Mean att-att attention  — success: {np.mean(att_success):.4f}  fail: {np.mean(att_fail):.4f}')
print(f'Mean att-def attention  — success: {np.mean(cross_success):.4f}  fail: {np.mean(cross_fail):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].boxplot([att_success, att_fail], labels=['Success', 'Fail'])
axes[0].set_title('Att–Att Edge Attention (last GAT layer)')
axes[0].set_ylabel('Mean attention weight')

axes[1].boxplot([cross_success, cross_fail], labels=['Success', 'Fail'])
axes[1].set_title('Att–Def Cross-Edge Attention (last GAT layer)')
axes[1].set_ylabel('Mean attention weight')

plt.tight_layout()
plt.savefig('gat_attention_analysis.png', dpi=150)
plt.show()